In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm

import os, json, random, copy, time, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from concurrent.futures import ThreadPoolExecutor
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT           = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR   = f"{OUT}/results"
FIGURES_DIR   = f"{OUT}/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- dataset -> local disk (parallel, resumable) ----
LOCAL_DATA = "/tmp/tea_leaf_local"
N_WORKERS  = 32

def _count_files(p):
    return sum(len(f) for _, _, f in os.walk(p)) if os.path.isdir(p) else 0

n_src = _count_files(TEA_LEAF_ROOT)
n_dst = _count_files(LOCAL_DATA)

if n_dst < n_src:
    t0 = time.time()
    tasks = []
    for root, _, files_ in os.walk(TEA_LEAF_ROOT):
        rel = os.path.relpath(root, TEA_LEAF_ROOT)
        dst_dir = LOCAL_DATA if rel == '.' else os.path.join(LOCAL_DATA, rel)
        os.makedirs(dst_dir, exist_ok=True)
        for f in files_:
            src, dst = os.path.join(root, f), os.path.join(dst_dir, f)
            if not os.path.exists(dst):
                tasks.append((src, dst))
    print(f"Copying {len(tasks)} files ({n_dst}/{n_src} local) with {N_WORKERS} workers...")
    def _cp(t):
        try:
            shutil.copyfile(t[0], t[1]); return True
        except Exception:
            return False
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        ok = list(tqdm(ex.map(_cp, tasks), total=len(tasks), desc="copy"))
    print(f" Copied {sum(ok)}/{len(tasks)} in {time.time()-t0:.0f}s")
else:
    print(f" Local cache complete ({n_dst} files)")

classes = sorted([d for d in os.listdir(LOCAL_DATA)
                  if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES     = len(classes)
CLASS_TO_IDX    = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]
print(f"Classes ({NUM_CLASSES}): {classes}")

dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn

def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads,
                                       C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = ((cls_q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()
    attn_full = ((q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x

dv1_last_attn.forward = _dv1_patched_forward

tf_dv1 = T.Compose([T.Resize((224,224)), T.ToTensor(),
                    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_train = T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.RandomRotation(20), T.ColorJitter(0.2,0.2,0.2), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_test = T.Compose([T.Resize((224,224)), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

DV1_GRID = 14
N_HEADS  = 6
topk     = max(1, int(0.1 * DV1_GRID * DV1_GRID))

with torch.no_grad():
    _ = dino_v1(torch.randn(1,3,224,224).to(device))
print(f"Attention hook OK: {_dv1_store['cls_attn'].shape}")

all_images = []
for cls_name in classes:
    cls_dir = os.path.join(LOCAL_DATA, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg','.jpeg','.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname),
                'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1,
            })

class SimpleDS(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs = imgs; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        return self.transform(Image.open(self.imgs[idx]['path']).convert('RGB')), self.labels[idx]

class TestDS(Dataset):
    def __init__(self, imgs, transform):
        self.imgs = imgs; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        return (self.transform(Image.open(self.imgs[idx]['path']).convert('RGB')),
                self.imgs[idx]['is_diseased'])

n_dis = sum(x['is_diseased'] for x in all_images)
print(f"\n Ready: {len(all_images)} images | Diseased: {n_dis}, Healthy: {len(all_images)-n_dis}")

In [ ]:
import os, shutil, sys
from google.colab import files

need = [f for f in ["make_qualitative_figures.py", "make_results_figures.py"]
        if not os.path.exists(f"/content/{f}")]
if need:
    print("Select:", need)
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)

for m in ["make_qualitative_figures", "make_results_figures"]:
    sys.modules.pop(m, None)

import make_qualitative_figures as qf
import make_results_figures as mrf
print("OK")

In [ ]:
import os

# qualitative (from your tea images) — needs GPU, ~2 min
qf.build_all(
    model      = dino_v1,
    tf         = tf_dv1,
    store      = _dv1_store,
    device     = device,
    all_images = all_images,
    classes    = classes,
    figdir     = FIGURES_DIR,
    grid       = DV1_GRID,
    sel_heads  = [0, 1],
    ann_json   = os.path.join(RESULTS_DIR, 'lesion_annotations.json'),
)

# quantitative (from your saved JSONs) — instant
mrf.build_all(RESULTS_DIR, FIGURES_DIR, class_names=classes)

print("\nAll figures:")
for f in sorted(os.listdir(FIGURES_DIR)):
    if f.endswith('.pdf'):
        print("  ", f)

In [ ]:
import os, shutil, sys
from google.colab import files
if not os.path.exists("/content/shape_confound.py"):
    print("Select shape_confound.py:")
    up = files.upload()
    src = list(up.keys())[0]
    if os.path.abspath(src) != "/content/shape_confound.py":
        shutil.move(src, "/content/shape_confound.py")
sys.modules.pop("shape_confound", None)
import shape_confound as sc

out, S, G, W, Y, CLS = sc.run(
    all_images = all_images,
    model = dino_v1, tf = tf_dv1, store = _dv1_store, device = device,
    grid = DV1_GRID, sel_heads = [0, 1], topk = topk,
    n_per_class = 250,
    results_dir = RESULTS_DIR,
)